**Assiignment no 6.Implementation of Vertical Federated Learning for Joint Feature-Based Prediction**

**Import Librares**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

**Load Dataset**

In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 59.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.51MB/s]


**Split Features Vertically**

dataset divided,first 392 features given to client A,Then last 392 given to client B

In [3]:
def split_features(x):

    x = x.view(-1, 28*28)

    x_a = x[:, :392]   # Client A
    x_b = x[:, 392:]   # Client B

    return x_a, x_b

**Client Models**

For each client feature vectors,embedding are created to sent it to server instead send entire data of images

In [4]:
class ClientA(nn.Module):

    def __init__(self):
        super(ClientA, self).__init__()
        self.fc = nn.Linear(392, 128)

    def forward(self, x):
        return torch.relu(self.fc(x))

In [5]:
class ClientB(nn.Module):

    def __init__(self):
        super(ClientB, self).__init__()
        self.fc = nn.Linear(392, 128)

    def forward(self, x):
        return torch.relu(self.fc(x))

**Server Model**

In [6]:
class Server(nn.Module):

    def __init__(self):
        super(Server, self).__init__()

        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, a, b):

        x = torch.cat((a, b), dim=1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

After sending Features vectores Model train at server side on both features

In [7]:
clientA = ClientA()
clientB = ClientB()
server = Server()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    list(clientA.parameters()) +
    list(clientB.parameters()) +
    list(server.parameters()),
    lr=0.001
)

**Traning Process**

In [8]:
for epoch in range(5):

    for images, labels in train_loader:

        x_a, x_b = split_features(images)

        # Client forward
        out_a = clientA(x_a)
        out_b = clientB(x_b)

        # Server combine
        output = server(out_a, out_b)

        loss = criterion(output, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.09646427631378174
Epoch: 1 Loss: 0.2608572840690613
Epoch: 2 Loss: 0.431616872549057
Epoch: 3 Loss: 0.027737325057387352
Epoch: 4 Loss: 0.0026697064749896526


Prediction

In [9]:
with torch.no_grad():

    correct = 0
    total = 0

    for images, labels in train_loader:

        x_a, x_b = split_features(images)

        out_a = clientA(x_a)
        out_b = clientB(x_b)

        output = server(out_a, out_b)

        _, predicted = torch.max(output, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 99.15166666666667
